# 195. 多 Agent Supervisor/Worker：委派与 Handoff 怎样设计？

> **面试问题：如何把任务、能力、预算、父 trace 和输入版本绑定到委派中，避免多 Agent 的越权、循环与陈旧结果？**

## 先给结论

不要把 Agent 面试题答成框架 API：先定义状态、动作、权限、预算、版本和可判定的终态，再讨论 prompt、模型和并发扩展。下面用受控内存数据手写最小协议；小规模断言只证明实现合同，不代表线上模型效果、权限体系或安全等级。

## 一手资料

- [AutoGen](https://arxiv.org/abs/2308.08155)
- [MetaGPT](https://arxiv.org/abs/2308.00352)
- [AgentDojo](https://arxiv.org/abs/2406.13352)

In [ ]:
notebook_contract = {"mode": "in-memory-demo", "oracle": "assertions", "production": "isolation-and-audit"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "in-memory-demo"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "assertions"  # 执行本行的状态、计算或校验逻辑。
assert "audit" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：委派是状态迁移，不是把 prompt 转发给另一个模型

多 Agent 的收益来自角色隔离、并行或专长，而不是角色数量本身。Supervisor 必须把任务、输入快照、授权能力、预算和父 trace 一并交给 Worker；Worker 返回的结果也必须标记来源和可验证状态。


In [ ]:
from dataclasses import dataclass  # 执行本行的状态、计算或校验逻辑。
@dataclass(frozen=True)  # 执行本行的状态、计算或校验逻辑。
class Task:  # 执行本行的状态、计算或校验逻辑。
    task_id: str  # 执行本行的状态、计算或校验逻辑。
    goal: str  # 执行本行的状态、计算或校验逻辑。
    capability: str  # 执行本行的状态、计算或校验逻辑。
    budget: int  # 执行本行的状态、计算或校验逻辑。
@dataclass(frozen=True)  # 执行本行的状态、计算或校验逻辑。
class Handoff:  # 执行本行的状态、计算或校验逻辑。
    parent_trace: str  # 执行本行的状态、计算或校验逻辑。
    task_id: str  # 执行本行的状态、计算或校验逻辑。
    worker: str  # 执行本行的状态、计算或校验逻辑。
task = Task("t-1", "查询订单", "orders.read", 2)  # 执行本行的状态、计算或校验逻辑。
assert task.capability == "orders.read"  # 执行本行的状态、计算或校验逻辑。
assert task.budget == 2  # 执行本行的状态、计算或校验逻辑。
assert task.task_id == "t-1"  # 执行本行的状态、计算或校验逻辑。


## 2. 角色能力：按 allow-list 决定谁能接什么任务

角色名称不是安全边界，能力集合才是。Supervisor 在委派前检查 Worker 的 allow-list；生产中还要把主体身份、租户、数据分级、速率和短期凭据纳入同一次策略判定。


In [ ]:
worker_capabilities = {"researcher": {"web.read"}, "order_worker": {"orders.read"}, "writer": set()}  # 执行本行的状态、计算或校验逻辑。
def can_delegate(worker, task):  # 执行本行的状态、计算或校验逻辑。
    return task.capability in worker_capabilities.get(worker, set())  # 执行本行的状态、计算或校验逻辑。
assert can_delegate("order_worker", task)  # 执行本行的状态、计算或校验逻辑。
assert not can_delegate("researcher", task)  # 执行本行的状态、计算或校验逻辑。
assert not can_delegate("missing", task)  # 执行本行的状态、计算或校验逻辑。


## 3. Supervisor：选择满足能力且剩余预算的 Worker

选择器不应只按模型置信度路由。这里按 allow-list 和当前负载挑选可执行 Worker；现实系统还应考虑队列、成本、区域、失败率、模型版本和是否允许并行。


In [ ]:
worker_load = {"order_worker": 1, "researcher": 0}  # 执行本行的状态、计算或校验逻辑。
def select_worker(task, loads):  # 执行本行的状态、计算或校验逻辑。
    candidates = [name for name in loads if can_delegate(name, task)]  # 执行本行的状态、计算或校验逻辑。
    if not candidates or task.budget <= 0:  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("没有可委派的 Worker 或预算已耗尽")  # 执行本行的状态、计算或校验逻辑。
    return min(candidates, key=lambda name: loads[name])  # 执行本行的状态、计算或校验逻辑。
chosen = select_worker(task, worker_load)  # 执行本行的状态、计算或校验逻辑。
assert chosen == "order_worker"  # 执行本行的状态、计算或校验逻辑。
assert worker_load[chosen] == 1  # 执行本行的状态、计算或校验逻辑。
assert task.budget > 0  # 执行本行的状态、计算或校验逻辑。


## 4. Worker：返回结构化观察，而不是未经标记的自然语言

Worker 输出应包含任务 id、输入版本、状态和值。否则 Supervisor 无法区分旧结果、不同任务结果与未经授权的数据。示例 Worker 只读固定订单，强调工具边界而非模型推理能力。


In [ ]:
orders = {"o-1": {"owner": "alice", "status": "paid", "version": 3}}  # 执行本行的状态、计算或校验逻辑。
def run_order_worker(handoff, order_id):  # 执行本行的状态、计算或校验逻辑。
    if handoff.worker != "order_worker":  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("错误 Worker 不能读取订单")  # 执行本行的状态、计算或校验逻辑。
    row = orders[order_id]  # 执行本行的状态、计算或校验逻辑。
    return {"task_id": handoff.task_id, "source_version": row["version"], "status": row["status"]}  # 执行本行的状态、计算或校验逻辑。
handoff = Handoff("trace-0", task.task_id, chosen)  # 执行本行的状态、计算或校验逻辑。
observation = run_order_worker(handoff, "o-1")  # 执行本行的状态、计算或校验逻辑。
assert observation["task_id"] == "t-1"  # 执行本行的状态、计算或校验逻辑。
assert observation["source_version"] == 3  # 执行本行的状态、计算或校验逻辑。
assert observation["status"] == "paid"  # 执行本行的状态、计算或校验逻辑。


## 5. 汇总：验收 task id、父 trace 和输入版本

Supervisor 不能把任何字符串拼入最终回答。它应验证结果属于当前 handoff，并在输入版本变化时拒绝旧观察。生产上还会校验 schema、签名、数据范围和 evidence 引用。


In [ ]:
def accept_observation(task, handoff, observation, current_version):  # 执行本行的状态、计算或校验逻辑。
    return handoff.task_id == task.task_id and observation["task_id"] == task.task_id and observation["source_version"] == current_version  # 执行本行的状态、计算或校验逻辑。
assert accept_observation(task, handoff, observation, 3)  # 执行本行的状态、计算或校验逻辑。
assert not accept_observation(task, handoff, observation, 4)  # 执行本行的状态、计算或校验逻辑。
assert not accept_observation(Task("t-2", "x", "orders.read", 1), handoff, observation, 3)  # 执行本行的状态、计算或校验逻辑。


## 6. 失败与循环：深度、预算和重复调用都要有上限

多个 Agent 可以相互委派，因而必须限制 delegation depth、总 token/工具预算和重复任务 key。没有这些 guard，系统会出现自我对话、级联成本或任务风暴。


In [ ]:
def next_budget(budget, depth, max_depth):  # 执行本行的状态、计算或校验逻辑。
    if budget <= 0 or depth >= max_depth:  # 执行本行的状态、计算或校验逻辑。
        return None  # 执行本行的状态、计算或校验逻辑。
    return {"budget": budget - 1, "depth": depth + 1}  # 执行本行的状态、计算或校验逻辑。
step = next_budget(2, 0, 2)  # 执行本行的状态、计算或校验逻辑。
assert step == {"budget": 1, "depth": 1}  # 执行本行的状态、计算或校验逻辑。
assert next_budget(0, 0, 2) is None  # 执行本行的状态、计算或校验逻辑。
assert next_budget(2, 2, 2) is None  # 执行本行的状态、计算或校验逻辑。


## 7. 评测：分解质量、委派正确性和最终状态分开看

只统计最终答对会掩盖越权、重复调用或错误角色拿到数据。示例指标把每条 trace 压成三个独立维度；生产中还要按任务类型、并发度、攻击输入和成本桶进行切片。


In [ ]:
def score_trace(events):  # 执行本行的状态、计算或校验逻辑。
    delegated = all(event["authorized"] for event in events)  # 执行本行的状态、计算或校验逻辑。
    bound = all(event["matched_task"] for event in events)  # 执行本行的状态、计算或校验逻辑。
    completed = events[-1]["terminal"] if events else False  # 执行本行的状态、计算或校验逻辑。
    return {"delegation": delegated, "binding": bound, "success": completed}  # 执行本行的状态、计算或校验逻辑。
score = score_trace([{"authorized": True, "matched_task": True, "terminal": False}, {"authorized": True, "matched_task": True, "terminal": True}])  # 执行本行的状态、计算或校验逻辑。
assert score["delegation"] is True  # 执行本行的状态、计算或校验逻辑。
assert score["binding"] is True  # 执行本行的状态、计算或校验逻辑。
assert score["success"] is True  # 执行本行的状态、计算或校验逻辑。


## 8. 制品：把角色、能力、模型和 trace 绑定

调试多 Agent 时最重要的是复放：谁在什么策略/模型/工具版本下收到了什么输入、向谁委派、为何接受或拒绝。示例摘要不记录敏感内容；生产日志必须做访问控制和保留期治理。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"supervisor": "v1", "worker": chosen, "capability": task.capability, "trace": handoff.parent_trace}  # 执行本行的状态、计算或校验逻辑。
fingerprint = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert artifact["worker"] == "order_worker"  # 执行本行的状态、计算或校验逻辑。
assert artifact["capability"] == "orders.read"  # 执行本行的状态、计算或校验逻辑。
assert len(fingerprint) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

回答时依次给出目标、状态合同、动作前校验、主路径、失败分支、指标、制品版本和生产替换点。可靠 Agent 不靠模型自述“完成”，而靠独立的状态 oracle、预算约束、审计和可复放 trace。
